In [ ]:
import requests
import pandas as pd
import json
from datetime import datetime, timedelta
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from threading import Lock

I_KEY = "PAR_formationdeanalysedum_e59db4c53b91e3a08a5bc0830c0195ad47124f62a46a30edcdb197bfd0a14d4b"
S_KEY = 'a32716290e384f574d265d8900fb72d635e9d79a580d883df96c6b126562ade6'

# Liste des départements français (métropole + DOM)
DEPARTEMENTS = [
    '01', '02', '03', '04', '05', '06', '07', '08', '09', '10',
    '11', '12', '13', '14', '15', '16', '17', '18', '19', '21',
    '22', '23', '24', '25', '26', '27', '28', '29', '2A', '2B',
    '30', '31', '32', '33', '34', '35', '36', '37', '38', '39',
    '40', '41', '42', '43', '44', '45', '46', '47', '48', '49',
    '50', '51', '52', '53', '54', '55', '56', '57', '58', '59',
    '60', '61', '62', '63', '64', '65', '66', '67', '68', '69',
    '70', '71', '72', '73', '74', '75', '76', '77', '78', '79',
    '80', '81', '82', '83', '84', '85', '86', '87', '88', '89',
    '90', '91', '92', '93', '94', '95',
    '971', '972', '973', '974', '976'
]

# Lock pour synchroniser l'affichage et les compteurs
print_lock = Lock()
stats_lock = Lock()

def get_access_token():
    """Obtenir le token d'accès"""
    token_url = 'https://entreprise.pole-emploi.fr/connexion/oauth2/access_token?realm=/partenaire'
    data = {
        'grant_type': 'client_credentials',
        'client_id': I_KEY,
        'client_secret': S_KEY,
        'scope': 'api_offresdemploiv2 o2dsoffre'
    }
    
    response = requests.post(token_url, data=data)
    
    if response.status_code == 200:
        return response.json()['access_token']
    else:
        raise Exception(f"Erreur token: {response.status_code} - {response.text}")

def fetch_offres_with_range(headers, params, max_offres=3149):
    """Récupère les offres avec pagination par range"""
    PAGE_SIZE = 150
    offres_batch = []
    start = 0
    MAX_RANGE = 3149
    
    API_FT = "https://api.francetravail.io/partenaire/offresdemploi/v2/offres/search"
    
    while start <= MAX_RANGE and len(offres_batch) < max_offres:
        end = min(start + PAGE_SIZE - 1, MAX_RANGE)
        
        params_copy = params.copy()
        params_copy['range'] = f'{start}-{end}'
        
        try:
            response = requests.get(API_FT, headers=headers, params=params_copy, timeout=10)
            if response.status_code not in (200, 206):
                break
            
            data = response.json()
            offres = data.get("resultats", [])
            
            if not offres:
                break
            
            offres_batch.extend(offres)
            
            if len(offres) < PAGE_SIZE:
                break
            
            start = end + 1
            time.sleep(0.1)  # Petite pause pour ne pas surcharger l'API
            
        except Exception as e:
            break
    
    return offres_batch

def fetch_period_simple(access_token, start_time, end_time, period_id, total_periods):
    """
    Récupère les offres d'une période SANS découpage départements (version rapide pour voir si nombre d'offres < 3150)
    """
    headers = {
        "Authorization": f'Bearer {access_token}',
        "Accept": "application/json"
    }
    
    min_date = start_time.strftime("%Y-%m-%dT%H:%M:%S") + "Z"
    max_date = end_time.strftime("%Y-%m-%dT%H:%M:%S") + "Z"
    
    params = {
        'minCreationDate': min_date,
        'maxCreationDate': max_date
    }
    
    offres_batch = fetch_offres_with_range(headers, params)
    
    with print_lock:
        print(f"Période {period_id}/{total_periods} ({start_time.strftime('%d/%m %H:%M')}) : {len(offres_batch)} offres")
    
    return {
        'offres': offres_batch,
        'hit_limit': len(offres_batch) >= 3149,
        'start_time': start_time,
        'end_time': end_time
    }

def fetch_dept_for_period(access_token, start_time, end_time, dept, period_label):
    headers = {
        "Authorization": f'Bearer {access_token}',
        "Accept": "application/json"
    }
    
    min_date = start_time.strftime("%Y-%m-%dT%H:%M:%S") + "Z"
    max_date = end_time.strftime("%Y-%m-%dT%H:%M:%S") + "Z"
    
    params = {
        'minCreationDate': min_date,
        'maxCreationDate': max_date,
        'departement': dept
    }
    
    offres_dept = fetch_offres_with_range(headers, params)
    
    return {
        'dept': dept,
        'offres': offres_dept,
        'hit_limit': len(offres_dept) >= 3149
    }

def fetch_all_offres_parallel(access_token, days_back=7, period_hours=12, max_workers=5):
        
    ALL_OFFRES = []
    seen_ids = set()
    
    # Obtenir minuit du jour actuel
    now = datetime.now()
    today_midnight = datetime(now.year, now.month, now.day, 0, 0, 0)
    
    # Calculer le nombre de périodes par jour
    periods_per_day = 24 // period_hours
    total_periods = days_back * periods_per_day
        
    # Définir les tranches horaires
    time_slots = []
    for i in range(periods_per_day):
        start_hour = (i * period_hours) + 1 if i == 0 else i * period_hours
        end_hour = ((i + 1) * period_hours) - 1
        if end_hour >= 24:
            end_hour = 23
        time_slots.append((start_hour, end_hour, 59))
    
    # Créer la liste de toutes les périodes à traiter
    periods_to_fetch = []
    period_counter = 0
    
    for day in range(days_back):
        current_day = today_midnight - timedelta(days=day)
        
        for start_hour, end_hour, end_minute in time_slots:
            period_counter += 1
            
            start_time = current_day.replace(hour=start_hour, minute=0, second=0)
            end_time = current_day.replace(hour=end_hour, minute=end_minute, second=59)
            
            if end_hour == 23 and end_minute == 59:
                end_time = current_day + timedelta(days=1, seconds=-1)
            
            periods_to_fetch.append({
                'id': period_counter,
                'start_time': start_time,
                'end_time': end_time
            })
    
    # ============ PHASE 1: EXTRACTION RAPIDE PARALLÈLE ============
    print(f"PHASE 1: Extraction de toutes les périodes")
    
    start_phase1 = time.time()
    periods_with_limit = []
    
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        # Soumettre toutes les tâches
        futures = {
            executor.submit(
                fetch_period_simple,
                access_token,
                p['start_time'],
                p['end_time'],
                p['id'],
                total_periods
            ): p for p in periods_to_fetch
        }
        
        # Récupérer les résultats au fur et à mesure
        for future in as_completed(futures):
            try:
                result = future.result()
                
                # Filtrer les doublons et ajouter
                nouvelles_offres = [
                    offre for offre in result['offres']
                    if offre.get('id') not in seen_ids
                ]
                
                for offre in nouvelles_offres:
                    seen_ids.add(offre.get('id'))
                
                ALL_OFFRES.extend(nouvelles_offres)
                
                # Si limite atteinte, noter pour phase 2
                if result['hit_limit']:
                    periods_with_limit.append({
                        'start_time': result['start_time'],
                        'end_time': result['end_time']
                    })
                
            except Exception as e:
                with print_lock:
                    print(f" Erreur: {e}")
    
    phase1_time = time.time() - start_phase1
    
    print(f"PHASE 1 TERMINÉE en {phase1_time:.1f}s")
    print(f"Total phase 1: {len(ALL_OFFRES)} offres uniques")
    print(f"{len(periods_with_limit)} période(s) ont atteint la limite de 3150\n")
    
    # ============ PHASE 2: RE-DÉCOUPAGE DÉPARTEMENTS (offres > 3150) ============
    if periods_with_limit:
        print(f"{'='*70}")
        print(f"PHASE 2: Re-découpage par départements pour {len(periods_with_limit)} période(s)")
                
        start_phase2 = time.time()
        
        # Départements prioritaires
        top_depts = ['75', '69', '13', '31', '59', '33', '44', '06', '92', '93', '94', '91', '78', '95', '77']
        other_depts = [d for d in DEPARTEMENTS if d not in top_depts]
        all_depts = top_depts + other_depts
        
        for period_idx, period in enumerate(periods_with_limit, 1):
            print(f"\n🔄 Période {period_idx}/{len(periods_with_limit)}: {period['start_time'].strftime('%d/%m %H:%M')}")
            print(f"   Re-découpage de {len(all_depts)} départements en parallèle...")
            
            period_label = f"{period['start_time'].strftime('%d/%m %H:%M')}"
            
            with ThreadPoolExecutor(max_workers=max_workers) as executor:
                futures = {
                    executor.submit(
                        fetch_dept_for_period,
                        access_token,
                        period['start_time'],
                        period['end_time'],
                        dept,
                        period_label
                    ): dept for dept in all_depts
                }
                
                dept_count = 0
                for future in as_completed(futures):
                    try:
                        result = future.result()
                        dept_count += 1
                        
                        if result['offres']:
                            # Filtrer les doublons
                            nouvelles_offres = [
                                offre for offre in result['offres']
                                if offre.get('id') not in seen_ids
                            ]
                            
                            for offre in nouvelles_offres:
                                seen_ids.add(offre.get('id'))
                            
                            ALL_OFFRES.extend(nouvelles_offres)
                            
                            if nouvelles_offres:
                                with print_lock:
                                    print(f"Dept {result['dept']}: +{len(nouvelles_offres)} nouvelles offres")

                    except Exception as e:
                        with print_lock:
                            print(f"   ❌ Erreur département: {e}")
            
            print(f" Période terminée. Total actuel: {len(ALL_OFFRES)} offres")
        
        phase2_time = time.time() - start_phase2
        
        print(f"PHASE 2 TERMINÉE en {phase2_time:.1f}s")
        print(f"{'='*70}")
    
    return ALL_OFFRES

def export_data(offres, filename_prefix="offres_emploi"):
    """Exporte les données en JSON"""
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    # JSON
    json_filename = f"{filename_prefix}_{timestamp}.json"
    
    with open(json_filename, "w", encoding="utf-8") as f:
        json.dump(offres, f, ensure_ascii=False, indent=4)
    print(f"JSON sauvegardé: {json_filename}")
    
# ============ PROGRAMME PRINCIPAL ============

if __name__ == "__main__":
    
    print("  EXTRACTION - FRANCE TRAVAIL API")

    # Paramètres
    DAYS_BACK = 3         # Nombre de jours à extraire
    PERIOD_HOURS = 24     # Taille des périodes en heures
    MAX_WORKERS = 10      # Nombre de threads parallèles (5-10 recommandés max) - Plus = plus rapide, mais attention au rate limiting
    
    print(f"Configuration:")
    print(f"Période: {DAYS_BACK} derniers jours") 
    print(f"Granularité: {PERIOD_HOURS}h par période")
    print(f"Multithread: {MAX_WORKERS} workers\n")
    
    total_start = time.time()
    
    try:
        # Authentification
        access_token = get_access_token()
        print(" Authentification réussie\n")
        
        # Extraction parallèle
        ALL_OFFRES = fetch_all_offres_parallel(
            access_token,
            days_back=DAYS_BACK,
            period_hours=PERIOD_HOURS,
            max_workers=MAX_WORKERS
        )
        
        total_time = time.time() - total_start
        
        print(f"🎉 EXTRACTION TOTALE TERMINÉE en {total_time:.1f}s ({total_time/60:.1f} min)")
        print(f" Total: {len(ALL_OFFRES)} offres uniques récupérées")
        print(f" Vitesse moyenne: {len(ALL_OFFRES)/total_time:.1f} offres/seconde\n")
        
        # Export
        export_data(ALL_OFFRES)
        
        print("Traitement terminé avec succès !")
        
    except Exception as e:
        print(f"\n Erreur: {e}")
        import traceback
        traceback.print_exc()

  EXTRACTION - FRANCE TRAVAIL API
Configuration:
Période: 3 derniers jours
Granularité: 24h par période
Multithread: 10 workers

 Authentification réussie

PHASE 1: Extraction de toutes les périodes
Période 3/3 (14/02 01:00) : 225 offres
Période 2/3 (15/02 01:00) : 3150 offres
Période 1/3 (16/02 01:00) : 3150 offres
PHASE 1 TERMINÉE en 22.7s
Total phase 1: 6525 offres uniques
2 période(s) ont atteint la limite de 3150

PHASE 2: Re-découpage par départements pour 2 période(s)

🔄 Période 1/2: 15/02 01:00
   Re-découpage de 101 départements en parallèle...
Dept 93: +48 nouvelles offres
Dept 06: +28 nouvelles offres
Dept 94: +61 nouvelles offres
Dept 91: +40 nouvelles offres
Dept 92: +148 nouvelles offres
Dept 31: +73 nouvelles offres
Dept 33: +125 nouvelles offres
Dept 59: +148 nouvelles offres
Dept 13: +119 nouvelles offres
Dept 44: +150 nouvelles offres
Dept 75: +163 nouvelles offres
Dept 69: +168 nouvelles offres
Dept 95: +41 nouvelles offres
Dept 78: +43 nouvelles offres
Dept 03: +1 n